In [1]:
import torch
import torch.nn as nn

import numpy as np

import mne
import matplotlib.pyplot as plt

mne.set_log_level("WARNING") # to suppress info messages

In [2]:
num_features = 40
kernel_size_temporal = 30
kernel_size_spatial = 14
kernel_size_avg_pool = 15
ffn_embedding_dim = 80
vocab_size = 3

temporal_conv = nn.Conv2d(in_channels=1,
                        out_channels=num_features,
                        kernel_size=(1, kernel_size_temporal),
                        padding="same")

spatial_conv = nn.Conv2d(in_channels=num_features,
                                       out_channels=num_features,
                                       kernel_size=(kernel_size_spatial, 1),
                                       padding="valid")

batch_norm1 = nn.BatchNorm2d(num_features)
batch_norm2 = nn.BatchNorm2d(num_features)

elu = nn.ELU()

avg_pool = nn.AvgPool2d(kernel_size=(1, kernel_size_avg_pool))

fc1 = nn.Linear(2400, ffn_embedding_dim)
fc2 = nn.Linear(ffn_embedding_dim, vocab_size)

softmax = nn.Softmax(dim=1)

In [3]:
x = torch.randn(32, 14, 900) # (N, C, T) where H is num_channels

x = x.unsqueeze(1) # (N, C, H, W) where C is 1 and H is num_channels

x = temporal_conv(x) # (32, 40, 14, 900)
x = batch_norm1(x) # (32, 40, 14, 900)
x = elu(x) # (32, 40, 14, 900)
print("before:", x.shape)

x = spatial_conv(x) # (32, 40, 1, 900)
x = batch_norm2(x) # (32, 40, 1, 900)
x = elu(x) # (32, 40, 1, 900)
print("after:", x.shape)

x = avg_pool(x) # (32, 40, 1, 60)
print("pooled:", x.shape)

x = torch.flatten(x, 1) # (32, 40 * 1 * 60) = (32, 2400)
print("flattened:", x.shape)

x = fc1(x)
x = fc2(x)
print("dense:", x.shape)

x = softmax(x) # (32, 3) - each row is a probability distribution across the 3 classes
print("softmax:", x.shape)
print(x)

/root/eeg/.venv/lib/python3.12/site-packages/torch/nn/modules/conv.py:543: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1031.)
  return F.conv2d(


before: torch.Size([32, 40, 14, 900])
after: torch.Size([32, 40, 1, 900])
pooled: torch.Size([32, 40, 1, 60])
flattened: torch.Size([32, 2400])
dense: torch.Size([32, 3])
softmax: torch.Size([32, 3])
tensor([[0.3302, 0.3058, 0.3640],
        [0.3419, 0.3016, 0.3565],
        [0.3328, 0.3052, 0.3620],
        [0.3416, 0.3659, 0.2924],
        [0.3356, 0.3209, 0.3435],
        [0.3496, 0.3099, 0.3405],
        [0.3361, 0.2885, 0.3753],
        [0.3552, 0.3191, 0.3256],
        [0.3432, 0.3029, 0.3539],
        [0.3148, 0.3028, 0.3825],
        [0.3749, 0.3272, 0.2979],
        [0.3682, 0.2816, 0.3502],
        [0.3711, 0.2941, 0.3349],
        [0.3292, 0.3124, 0.3584],
        [0.3262, 0.3318, 0.3420],
        [0.3797, 0.2926, 0.3277],
        [0.3345, 0.3037, 0.3618],
        [0.3184, 0.3140, 0.3676],
        [0.3495, 0.2796, 0.3709],
        [0.3153, 0.3126, 0.3721],
        [0.3432, 0.3208, 0.3360],
        [0.3247, 0.3242, 0.3511],
        [0.3453, 0.3104, 0.3443],
        [0.3761, 0

In [4]:
raw = mne.io.read_raw_edf("/var/log/thavamount/eeg_dataset/motor_eeg/1.0.0/S001/S001R03.edf", preload=True)

In [5]:
events, event_ids = mne.events_from_annotations(raw)
epochs = mne.Epochs(raw, events=events)

In [6]:
print(raw.get_data().shape) # (64, 20000)
print(events.shape) # every row is [sample, 0, event_id]
print(epochs["2"].get_data().shape) # corresponding to right hand, (n_epochs, C, T)
print(epochs.get_data().shape) # (29, 64, 113)

(64, 20000)
(30, 3)
(8, 64, 113)
(29, 64, 113)


In [7]:
# we want to take the differences between the first column numbers in events to get the time duration of each epoch
event_space = events[1:, 0] - events[:-1, 0]
print(event_space)

[672 656 672 656 672 656 672 656 672 656 672 656 672 656 672 656 672 656
 672 656 672 656 672 656 672 656 672 656 672]


In [8]:
events, event_ids = mne.events_from_annotations(raw)
events = np.delete(events, 1, axis=1) # every row: (sample, event_id)
print(events)
print(epochs.get_data())

[[    0     1]
 [  672     3]
 [ 1328     1]
 [ 2000     2]
 [ 2656     1]
 [ 3328     2]
 [ 3984     1]
 [ 4656     3]
 [ 5312     1]
 [ 5984     3]
 [ 6640     1]
 [ 7312     2]
 [ 7968     1]
 [ 8640     2]
 [ 9296     1]
 [ 9968     3]
 [10624     1]
 [11296     2]
 [11952     1]
 [12624     3]
 [13280     1]
 [13952     3]
 [14608     1]
 [15280     2]
 [15936     1]
 [16608     2]
 [17264     1]
 [17936     3]
 [18592     1]
 [19264     2]]
[[[-1.50000000e-05 -6.00000000e-06  4.00000000e-06 ... -5.40000000e-05
   -4.80000000e-05 -6.00000000e-05]
  [ 2.90909091e-06 -4.09090909e-06  2.90909091e-06 ... -3.70909091e-05
   -3.50909091e-05 -4.70909091e-05]
  [ 3.48484848e-06  9.48484848e-06  6.48484848e-06 ... -2.85151515e-05
   -3.15151515e-05 -4.25151515e-05]
  ...
  [ 1.71818182e-05  1.01818182e-05  1.41818182e-05 ... -1.38181818e-05
   -1.58181818e-05  1.81818182e-07]
  [ 9.03030303e-06 -2.96969697e-06 -6.96969697e-06 ... -4.09696970e-05
   -6.09696970e-05 -4.19696970e-05]
  [-4.48

In [9]:
# checking why some epochs are dropped
print(len(events))
print(len(epochs.events))
print(epochs.drop_log)

30
29
(('NO_DATA',), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), ())


In [63]:
from tqdm import tqdm
from eeg.eeg_data import EEGCNN, HandDatasetCNN

In [23]:
hand_dataset_cnn: HandDatasetCNN = HandDatasetCNN(num_folders=5)
model = EEGCNN(seq_len=hand_dataset_cnn.eeg_chunks.shape[-1])

state_dict = torch.load("/var/log/thavamount/eeg_ckpts/cnn/eeg_basic_cnn_epoch_1000.pth", map_location="cuda")
model.load_state_dict(state_dict["model"])

<All keys matched successfully>

In [22]:
eeg_chunk, label_chunk = hand_dataset_cnn[0]
print(eeg_chunk.shape)
print(label_chunk.shape)

(64, 113)
torch.Size([])


In [67]:
@torch.no_grad()
def inference(
    model: torch.nn.Module, 
    dataset: HandDatasetCNN,
    num_runs: int = 1,
    device: str = "cuda"
) -> torch.Tensor:

    model.to(device)
    model.eval()

    for i in tqdm(range(num_runs)):
        eeg_chunk, label_chunk = dataset[i]
        print(eeg_chunk.shape)
        print(label_chunk.shape)

        eeg_chunk = torch.tensor(eeg_chunk).to(device).float()
        label_chunk = label_chunk.to(device).unsqueeze(0)

        output = model(eeg_chunk).squeeze(0)

    label = torch.argmax(output).item()

    return label

In [68]:
out = inference(model, hand_dataset_cnn, num_runs=3)
print(out)

  0%|          | 0/3 [00:00<?, ?it/s]

(64, 113)
torch.Size([])


RuntimeError: Given groups=1, weight of size [40, 1, 1, 30], expected input[1, 64, 1, 114] to have 1 channels, but got 64 channels instead